# 02 - Load the tennis data into BigQuery
Produces the table tennis_raw.wta_matches (unedited, every column STRING)
It is an external table: BigQuery reads the compressed raw file in the bucket directly, so the raw data is stored once (in the bucket) and not copied into BigQuery.
Run after 01_ingest_tennis

## Install the pinned libraries

In [ ]:

# Read from a git tag rather than from main, so a later re-run installs exactly the same versions.
CODE_VERSION = "code-v1"      # the git tag of the code being run: create it on GitHub before running
REPO = "katiedevane/tennis-project"
REQUIREMENTS_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

# Shared provenance helpers (code version, environment, index of data versions), from the same tag
PROVENANCE_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/provenance.py"
!wget -q -O provenance.py $PROVENANCE_URL

## Sign in and load settings from Colab Secrets
Switch on Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Record the code version and set the random seed
The git tag, commit, Python environment and random seed of this run are added to every manifest this notebook writes (see provenance.py).
`SEED` is the one project-wide seed: `start_run` applies it, and it should be passed wherever a function takes a seed (e.g. `random_state=SEED`).

In [ ]:
from provenance import start_run, register, table_version, sha256, utc_stamp, SEED

NOTEBOOK = "02_load_tennis_bigquery"
RUN_INFO = start_run(f"https://github.com/{REPO}", CODE_VERSION, NOTEBOOK)

## Find the commit and checksums the ingest notebook recorded

In [ ]:
import json
import pandas as pd
from google.cloud import bigquery, storage

gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET_NAME)
bq = bigquery.Client(project=PROJECT_ID, location=REGION)

names = sorted(b.name for b in gcs.list_blobs(BUCKET_NAME, prefix="manifests/wta_matches_"))
m = json.loads(bucket.blob(names[-1]).download_as_text())      # the latest manifest

assert "columns" in m, "The latest manifest is from the old ingest notebook: run 01_ingest_tennis_revised first"
COMMIT = m["commit"]
COLUMNS = m["columns"]
RAW_FILE = m["files"][0]          # the one combined, compressed file
print("manifests found:", len(names), "| using the latest | commit:", COMMIT[:8], "| file:", RAW_FILE["file"])

## Check the stored file is the one the manifest recorded, and has the recorded columns
BigQuery matches columns by position, so need to match
(The check that all season files share one header now runs in 01, before they are combined.)

In [ ]:
import gzip, hashlib

stored = bucket.blob(RAW_FILE["file"]).download_as_bytes()
assert hashlib.sha256(stored).hexdigest() == RAW_FILE["sha256"], "The stored file differs from the manifest: stop and investigate"

header = gzip.decompress(stored).decode("utf-8-sig").splitlines()[0]
assert header.split(",") == COLUMNS, "The file's header differs from the columns in the manifest"
print(len(COLUMNS), "columns:", COLUMNS)

## Create the external table
Every column is read as text.
The table holds no copy of the data: queries read the compressed file in the bucket. The earlier loaded copy is replaced, so the raw rows are no longer stored twice.

In [ ]:
TABLE = f"{PROJECT_ID}.tennis_raw.wta_matches"
uri = f"gs://{BUCKET_NAME}/{RAW_FILE['file']}"

ext = bigquery.ExternalConfig("CSV")
ext.source_uris = [uri]
ext.schema = [bigquery.SchemaField(c, "STRING") for c in COLUMNS]
ext.options.skip_leading_rows = 1
ext.compression = "GZIP"
ext.max_bad_records = 0          # any malformed row makes queries fail loudly

table = bigquery.Table(TABLE)
table.external_data_configuration = ext
table.description = f"Raw WTA matches (commit {COMMIT[:8]}), read in place from {uri}. Every column STRING."
bq.delete_table(TABLE, not_found_ok=True)      # removes the old loaded copy (or the external table of an earlier run)
bq.create_table(table)
print("External table:", TABLE, "->", uri)

## Check: does BigQuery read exactly what the download recorded?

In [ ]:
n = int(bq.query(f"SELECT COUNT(*) AS n FROM `{TABLE}`",
                 job_config=bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)).to_dataframe()["n"][0])

check = pd.DataFrame({"manifest_rows": {e["file"]: e["rows"] for e in m["source_files"]}})
print(check)
print("manifest total:", check["manifest_rows"].sum(), "| combined file:", RAW_FILE["rows"], "| rows read by BigQuery:", n)
assert n == RAW_FILE["rows"] == check["manifest_rows"].sum(), "BigQuery does not read every row of the file"

## Record the table version
Adds the table's row count and content fingerprint to the index of data versions (`manifests/data_versions.json`), with the code version and the file it reads.

In [ ]:
raw_version = table_version(bq, TABLE)
register(bucket, "tennis_raw.wta_matches", raw_version["fingerprint"], outputs=[raw_version],
         parents=[RAW_FILE], manifest_path=names[-1])